In [ ]:
from google.colab import files
import pandas as pd
import io

uploaded = files.upload()

datasets = {}

for filename, content in uploaded.items():
    name = filename.lower()

    if name.startswith("customer_info"):
        datasets["customers"] = pd.read_csv(io.BytesIO(content))
    elif name.startswith("usage_data"):
        datasets["usage"] = pd.read_csv(io.BytesIO(content))
    elif name.startswith("churn_labels"):
        datasets["labels"] = pd.read_csv(io.BytesIO(content))

required = {"customers", "usage", "labels"}
missing = required - datasets.keys()

if missing:
    raise ValueError(f"Missing files for: {', '.join(sorted(missing))}")

customers = datasets["customers"]
usage = datasets["usage"]
labels = datasets["labels"]

for name, table in datasets.items():
    print(f"{name}: {len(table):,} rows")
    print("Columns:", ", ".join(table.columns))
    print()


Saving churn_labels.csv to churn_labels.csv
Saving customer_info.csv to customer_info.csv
Saving usage_data.csv to usage_data.csv
labels: 10,000 rows
Columns: CustomerID, Churn

customers: 10,200 rows
Columns: CustomerID, Age, Gender, Region, ContractType, MonthlyCharges, SignupDate

usage: 120,000 rows
Columns: CustomerID, Month, CallMinutes, DataUsageGB, SMSCount, Complaints



In [ ]:
# Work on copies so the uploaded data stays unchanged
customers_clean = customers.copy()
usage_clean = usage.copy()
labels_clean = labels.copy()

# Clean text fields
for table in [customers_clean, usage_clean, labels_clean]:
    for column in table.select_dtypes(include="object").columns:
        table[column] = table[column].str.strip()

# Convert dates
customers_clean["SignupDate"] = pd.to_datetime(
    customers_clean["SignupDate"], format="%Y-%m-%d", errors="raise"
)
usage_clean["Month"] = pd.to_datetime(
    usage_clean["Month"], format="%Y-%m-%d", errors="raise"
)

# Keep the earliest signup record per customer
customers_clean = (
    customers_clean
    .sort_values("SignupDate", kind="stable")
    .drop_duplicates("CustomerID", keep="first")
    .reset_index(drop=True)
)

# Attach signup date and exclude usage before signup
usage_clean = usage_clean.merge(
    customers_clean[["CustomerID", "SignupDate"]],
    on="CustomerID",
    how="left",
    validate="many_to_one"
)

assert usage_clean["SignupDate"].notna().all(), "Missing signup dates"

before_signup = usage_clean["Month"] < usage_clean["SignupDate"]
removed_rows = int(before_signup.sum())

usage_clean = usage_clean.loc[~before_signup].copy()

print(f"Unique customers: {len(customers_clean):,}")
print(f"Usage rows excluded before signup: {removed_rows:,}")
print(f"Valid usage rows: {len(usage_clean):,}")
print(f"Churn-labelled customers: {labels_clean['Churn'].sum():,}")


Unique customers: 10,000
Usage rows excluded before signup: 2,261
Valid usage rows: 117,739
Churn-labelled customers: 786


In [3]:
# Summarize usage for each customer
usage_summary = (
    usage_clean.groupby("CustomerID", as_index=False)
    .agg(
        MonthsObserved=("Month", "nunique"),
        AverageCallMinutes=("CallMinutes", "mean"),
        AverageDataUsageGB=("DataUsageGB", "mean"),
        TotalComplaints=("Complaints", "sum"),
        AverageMonthlyComplaints=("Complaints", "mean")
    )
)

# Combine customer details, usage, and churn labels
model_data = (
    customers_clean
    .merge(
        usage_summary,
        on="CustomerID",
        how="left",
        validate="one_to_one"
    )
    .merge(
        labels_clean[["CustomerID", "Churn"]],
        on="CustomerID",
        how="left",
        validate="one_to_one"
    )
)

assert model_data["CustomerID"].is_unique
assert model_data["Churn"].notna().all()

print(f"Customer rows: {len(model_data):,}")
print(f"Churn-labelled customers: {model_data['Churn'].sum():,}")

display(model_data.head())

Customer rows: 10,000
Churn-labelled customers: 786


,CustomerID,Age,Gender,Region,ContractType,MonthlyCharges,SignupDate,MonthsObserved,AverageCallMinutes,AverageDataUsageGB,TotalComplaints,AverageMonthlyComplaints,Churn
0,CUST_3769,NaN,Female,Urban,Monthly,46.32,2018-01-01,12,635.257500,8.630833,5,0.416667,0
1,CUST_6395,53.0,Male,Urban,Prepaid,78.47,2018-01-01,12,463.667500,9.449167,6,0.500000,0
2,CUST_5837,NaN,Male,Rural,Monthly,43.97,2018-01-02,12,485.919167,9.373333,6,0.500000,0
3,CUST_9565,NaN,Male,Suburban,Prepaid,60.35,2018-01-02,12,509.222500,9.820833,4,0.333333,0
4,CUST_308,37.0,Male,Urban,Monthly,42.28,2018-01-03,12,307.675833,8.905833,10,0.833333,0


In [4]:
from sklearn.model_selection import train_test_split

# Information the model will use
feature_columns = [
    "Age",
    "Gender",
    "Region",
    "ContractType",
    "MonthlyCharges",
    "MonthsObserved",
    "AverageCallMinutes",
    "AverageDataUsageGB",
    "TotalComplaints",
    "AverageMonthlyComplaints"
]

X = model_data[feature_columns].copy()
y = model_data["Churn"].astype(int)

# Keep similar proportions of churned customers in both groups
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training customers: {len(X_train):,}")
print(f"Testing customers: {len(X_test):,}")
print(f"Churn-labelled customers in training: {y_train.sum():,}")
print(f"Churn-labelled customers in testing: {y_test.sum():,}")

Training customers: 8,000
Testing customers: 2,000
Churn-labelled customers in training: 629
Churn-labelled customers in testing: 157


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Treat negative monthly charges as missing
X_train = X_train.copy()
X_test = X_test.copy()

for table in [X_train, X_test]:
    table["MonthlyCharges"] = table["MonthlyCharges"].mask(
        table["MonthlyCharges"] < 0
    )

categorical_columns = ["Gender", "Region", "ContractType"]
numeric_columns = [
    column for column in X_train.columns
    if column not in categorical_columns
]

# Fill missing numbers using the training median, then scale
numeric_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
    ("scaler", StandardScaler())
])

# Fill missing text and convert categories into numeric columns
categorical_steps = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_steps, numeric_columns),
    ("categorical", categorical_steps, categorical_columns)
])

# Learn preparation rules from training data only
X_train_ready = preprocessor.fit_transform(X_train)

# Apply those same rules to testing data
X_test_ready = preprocessor.transform(X_test)

print("Training data shape:", X_train_ready.shape)
print("Testing data shape:", X_test_ready.shape)
print("Data preparation complete.")

Training data shape: (8000, 17)
Testing data shape: (2000, 17)
Data preparation complete.


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

# Train using training customers only
model = LogisticRegression(
    max_iter=2000,
    random_state=42
)
model.fit(X_train_ready, y_train)

# Evaluate on customers excluded from training
test_scores = model.predict_proba(X_test_ready)[:, 1]
test_predictions = model.predict(X_test_ready)

print(f"ROC-AUC: {roc_auc_score(y_test, test_scores):.3f}")
print(
    f"Average precision: "
    f"{average_precision_score(y_test, test_scores):.3f}"
)
print(f"Baseline churn share: {y_test.mean():.3f}")

print("\nClassification report:")
print(classification_report(
    y_test,
    test_predictions,
    target_names=["Not churned", "Churned"],
    zero_division=0
))

print("Confusion matrix:")
print(confusion_matrix(y_test, test_predictions))

ROC-AUC: 0.505
Average precision: 0.085
Baseline churn share: 0.079

Classification report:
              precision    recall  f1-score   support

 Not churned       0.92      1.00      0.96      1843
     Churned       0.00      0.00      0.00       157

    accuracy                           0.92      2000
   macro avg       0.46      0.50      0.48      2000
weighted avg       0.85      0.92      0.88      2000

Confusion matrix:
[[1843    0]
 [ 157    0]]


In [7]:
train_scores = model.predict_proba(X_train_ready)[:, 1]

print(
    f"Training ROC-AUC: "
    f"{roc_auc_score(y_train, train_scores):.3f}"
)
print(
    f"Testing ROC-AUC: "
    f"{roc_auc_score(y_test, test_scores):.3f}"
)
print(f"Lowest test score: {test_scores.min():.3f}")
print(f"Highest test score: {test_scores.max():.3f}")

Training ROC-AUC: 0.540
Testing ROC-AUC: 0.505
Lowest test score: 0.000
Highest test score: 0.126


In [8]:
# Compare the latest 3 months with the preceding 3 months
usage_trends = usage_clean.copy()
usage_trends["Period"] = usage_trends["Month"].dt.to_period("M")
latest_month = usage_trends["Period"].max()

recent = (
    usage_trends[
        usage_trends["Period"] >= latest_month - 2
    ]
    .groupby("CustomerID")[["CallMinutes", "DataUsageGB"]]
    .mean()
)

previous = (
    usage_trends[
        usage_trends["Period"].between(
            latest_month - 5, latest_month - 3
        )
    ]
    .groupby("CustomerID")[["CallMinutes", "DataUsageGB"]]
    .mean()
)

# Percentage change; leave undefined comparisons as missing
usage_change = (
    recent.subtract(previous)
    .div(previous.where(previous > 0))
    .mul(100)
    .rename(columns={
        "CallMinutes": "CallChangePct",
        "DataUsageGB": "DataChangePct"
    })
)

# Investigate training customers only
training_check = (
    model_data.loc[X_train.index, ["CustomerID", "Churn"]]
    .merge(
        usage_change,
        on="CustomerID",
        how="left",
        validate="one_to_one"
    )
)

print("Average usage change (%) — training customers only:")
print(
    training_check.groupby("Churn")[
        ["CallChangePct", "DataChangePct"]
    ].mean().round(2)
)

Average usage change (%) — training customers only:
       CallChangePct  DataChangePct
Churn                              
0              -4.66          -0.99
1              -4.32          -0.82


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.base import clone

forest_pipeline = Pipeline([
    ("preparation", clone(preprocessor)),
    ("model", RandomForestClassifier(
        n_estimators=150,
        max_depth=6,
        min_samples_leaf=20,
        random_state=42,
        n_jobs=-1
    ))
])

cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

results = cross_validate(
    forest_pipeline,
    X_train,
    y_train,
    cv=cv,
    scoring={
        "auc": "roc_auc",
        "ap": "average_precision"
    }
)

print(f"Random Forest CV ROC-AUC: {results['test_auc'].mean():.3f}")
print(f"Random Forest CV average precision: {results['test_ap'].mean():.3f}")
print(f"Baseline churn share: {y_train.mean():.3f}")

Random Forest CV ROC-AUC: 0.494
Random Forest CV average precision: 0.081
Baseline churn share: 0.079


## Churn Modelling Results

- Prepared 10,000 unique customers.
- Excluded 2,261 usage records dated before customer signup.
- Used 8,000 customers for training and 2,000 for testing.
- Filled missing values and encoded categories using training data only.

### Results
- Logistic Regression test ROC-AUC: 0.505.
- Random Forest training cross-validation ROC-AUC: 0.494.
- Recent usage changes showed similar averages for both churn groups.

### Conclusion
The tested features and models did not reliably distinguish churned
customers. Prediction scores were therefore not added to Power BI.

Further work should confirm when churn labels were recorded relative
to usage and investigate additional predictors.